# Detección de Fraude — Pipeline Completo

**Pipeline perfeccionado con:**
- Feature Engineering v3 (ratio, temporales, geográficas, sesión, red, interacciones)
- KNNImputer para valores faltantes
- LightGBM + XGBoost con mejores hiperparámetros
- Ensemble calibrado (promedio ponderado óptimo)
- Threshold F2

**Dataset:** `dataset_fraude.csv` (v1) o `dataset_fraude_v2.csv` (v2).
**Target:** `IS_FRAUD` (binario).

> Ejecutar desde la raíz del proyecto (`C:\Dev\NovaPay_ML`).

## Configuración

Cambiar `DATASET` para elegir versión de datos.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score,
    precision_score, recall_score, make_scorer, fbeta_score)
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import lightgbm as lgb
sys.path.append(str(Path.cwd()))
from scripts.feature_engineering import FeatureEngineer
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

# ============================================================
# CONFIG
# ============================================================
# Elegir dataset: 'v1' (original) o 'v2'
DATASET = 'v2'

if DATASET == 'v2':
    DATA_PATH = Path.cwd() / 'scripts' / 'synthetic_data' / 'dataset_fraude_v2.csv'
    LABEL = 'v2'
else:
    DATA_PATH = Path.cwd() / 'Notebooks' / 'data' / 'dataset_fraude.csv'
    LABEL = 'v1 (original)'

print(f'Dataset: {LABEL}')
print(f'Path: {DATA_PATH}')

## 1. Carga y exploración

In [ ]:
df = pd.read_csv(DATA_PATH)
other_target = [c for c in ['IS_FRAUD', 'IMPACTO_FRAUDE'] if c != 'IS_FRAUD']
if other_target:
    df = df.drop(columns=other_target, errors='ignore')

print(f'Filas: {df.shape[0]:,}  Columnas: {df.shape[1]}')
print(f'Fraude: {df["IS_FRAUD"].sum():,} / {len(df):,} ({df["IS_FRAUD"].mean()*100:.2f}%)')
print(f'\nColumnas:')
for c in df.columns:
    dtype = df[c].dtype
    nunique = df[c].nunique() if dtype == 'object' else '-'
    miss = df[c].isna().sum()
    print(f'  {c:35s}  {str(dtype):10s}  únicos={str(nunique):>5s}  miss={miss}')

## 2. Feature Engineering v3

**Nuevas features añadidas:**

**Sesión (burst detection):**
- `txn_por_minuto`: transacciones por minuto en sesión actual
- `burst_rapido`: >5 tx última hora Y última tx < 5 min
- `alta_velocidad`: >3 tx última hora O última tx < 30s
- `monto_velocidad`: importe × tx_última_hora
- `tiempo_ultima_bin`: tiempo desde última tx en buckets

**Desviación temporal (por cliente):**
- `diff_hora_cliente`: desviación absoluta de hora media del cliente
- `diff_importe_cliente`: desviación relativa del importe medio del cliente
- `ratio_actividad_cliente`: tx_última_hora vs media diaria del cliente

**Red:**
- `frecuencia_destino`: cuántas veces aparece cada cuenta destino
- `foreign_unknown_device`: país extranjero + dispositivo desconocido
- `night_velocity`: noche + alta velocidad
- `high_ratio_redondeado`: importe >85% límite + redondeado

In [ ]:
fe = FeatureEngineer(encode_target='IS_FRAUD', random_state=42)
X = fe.fit_transform(df)
y = X.pop('IS_FRAUD').values

print(f'Features totales: {X.shape[1]}')
print(f'Features numéricas: {len(X.select_dtypes(include=[np.number]).columns)}')
print()
for c in sorted(X.columns):
    print(f'  {c}')

## 3. Train / Validation / Test Split + KNNImputer

60/20/20 estratificado. KNNImputer después de escalado.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train:  {X_train.shape[0]:,}  ({y_train.mean()*100:.2f}% fraude)')
print(f'Val:    {X_val.shape[0]:,}  ({y_val.mean()*100:.2f}% fraude)')
print(f'Test:   {X_test.shape[0]:,}  ({y_test.mean()*100:.2f}% fraude)')

In [ ]:
num_feats = X_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
X_train_s = X_train.copy()
X_train_s[num_feats] = scaler.fit_transform(X_train[num_feats])
X_val_s = X_val.copy()
X_val_s[num_feats] = scaler.transform(X_val[num_feats])
X_test_s = X_test.copy()
X_test_s[num_feats] = scaler.transform(X_test[num_feats])
print(f'Escalado: {len(num_feats)} features numéricas')

imputer = KNNImputer(n_neighbors=5)
X_train_s[num_feats] = imputer.fit_transform(X_train_s[num_feats])
X_val_s[num_feats] = imputer.transform(X_val_s[num_feats])
X_test_s[num_feats] = imputer.transform(X_test_s[num_feats])

train_miss = np.isnan(X_train_s[num_feats]).sum().sum()
print(f'Missing después de imputación (train): {train_miss}')
print('OK')

## 4. LightGBM

Entrenamiento con mejores hiperparámetros según dataset.

In [ ]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

if DATASET == 'v2':
    lgb_params = {
        'learning_rate': 0.1, 'min_child_samples': 100,
        'n_estimators': 200, 'num_leaves': 15,
        'reg_lambda': 10, 'scale_pos_weight': float(scale_pos),
        'subsample': 0.8,
    }
else:
    lgb_params = {
        'learning_rate': 0.05, 'min_child_samples': 20,
        'n_estimators': 200, 'num_leaves': 15,
        'reg_lambda': 10, 'scale_pos_weight': float(scale_pos),
        'subsample': 0.8,
    }

lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1, **lgb_params)
lgb_model.fit(X_train_s, y_train)

yprob_lgb_val = lgb_model.predict_proba(X_val_s)[:, 1]
yprob_lgb_test = lgb_model.predict_proba(X_test_s)[:, 1]

prauc_lgb_val = average_precision_score(y_val, yprob_lgb_val)
auc_lgb_val = roc_auc_score(y_val, yprob_lgb_val)
print(f'LightGBM Val:  PR-AUC={prauc_lgb_val:.4f}  AUC-ROC={auc_lgb_val:.4f}')

## 5. XGBoost

In [ ]:
if DATASET == 'v2':
    xgb_params = {
        'learning_rate': 0.05, 'max_depth': 3,
        'min_child_weight': 1, 'n_estimators': 200,
        'reg_lambda': 0, 'scale_pos_weight': float(scale_pos),
        'subsample': 1.0,
    }
else:
    xgb_params = {
        'learning_rate': 0.05, 'max_depth': 3,
        'min_child_weight': 1, 'n_estimators': 200,
        'reg_lambda': 0, 'scale_pos_weight': float(scale_pos),
        'subsample': 1.0,
    }

xgb_model = xgb.XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss', **xgb_params)
xgb_model.fit(X_train_s, y_train)

yprob_xgb_val = xgb_model.predict_proba(X_val_s)[:, 1]
yprob_xgb_test = xgb_model.predict_proba(X_test_s)[:, 1]

prauc_xgb_val = average_precision_score(y_val, yprob_xgb_val)
auc_xgb_val = roc_auc_score(y_val, yprob_xgb_val)
print(f'XGBoost Val:   PR-AUC={prauc_xgb_val:.4f}  AUC-ROC={auc_xgb_val:.4f}')

## 6. Calibración

Se calibran ambos modelos con 3-fold CV. Si no mejora PR-AUC, se usan probabilidades raw.

In [ ]:
cal_lgb = CalibratedClassifierCV(lgb_model, cv=3, method='sigmoid')
cal_lgb.fit(X_train_s, y_train)
cal_xgb = CalibratedClassifierCV(xgb_model, cv=3, method='sigmoid')
cal_xgb.fit(X_train_s, y_train)

use_cal = False
for raw_model, cal_model, name in [(lgb_model, cal_lgb, 'LightGBM'), (xgb_model, cal_xgb, 'XGBoost')]:
    yprob_raw = raw_model.predict_proba(X_val_s)[:, 1]
    yprob_cal = cal_model.predict_proba(X_val_s)[:, 1]
    pauc_raw = average_precision_score(y_val, yprob_raw)
    pauc_cal = average_precision_score(y_val, yprob_cal)
    diff = pauc_cal - pauc_raw
    if diff > 0.01:
        use_cal = True
    print(f'{name}:  Raw={pauc_raw:.4f}  Cal={pauc_cal:.4f}  Diff={diff:+.4f}')

if use_cal:
    print('\n>> Usando probabilidades calibradas')
    yprob_lgb_val = cal_lgb.predict_proba(X_val_s)[:, 1]
    yprob_xgb_val = cal_xgb.predict_proba(X_val_s)[:, 1]
    yprob_lgb_test = cal_lgb.predict_proba(X_test_s)[:, 1]
    yprob_xgb_test = cal_xgb.predict_proba(X_test_s)[:, 1]
else:
    print('\n>> Usando probabilidades raw (calibración no mejora)')

## 7. Ensemble Calibrado (LightGBM + XGBoost)

Promedio ponderado: `yprob_ens = w * yprob_lgb + (1-w) * yprob_xgb`

Peso óptimo `w` buscado en validación maximizando PR-AUC.

In [ ]:
weights = np.linspace(0, 1, 101)
best_w = 0.5
best_prauc = 0
results_w = []

for w in weights:
    yprob_ens = w * yprob_lgb_val + (1 - w) * yprob_xgb_val
    prauc = average_precision_score(y_val, yprob_ens)
    results_w.append({'w': w, 'PR-AUC': prauc})
    if prauc > best_prauc:
        best_prauc = prauc
        best_w = w

print(f'Peso óptimo: w={best_w:.3f} (LightGBM) + {1-best_w:.3f} (XGBoost)')
print(f'PR-AUC ensemble (val): {best_prauc:.4f}')
print(f'LightGBM solo (val):   {prauc_lgb_val:.4f}')
print(f'XGBoost solo (val):    {prauc_xgb_val:.4f}')

df_w = pd.DataFrame(results_w)
plt.figure(figsize=(10, 4))
plt.plot(df_w['w'], df_w['PR-AUC'], 'b-', lw=2)
plt.axvline(x=best_w, color='r', linestyle='--', alpha=0.5)
plt.xlabel('Peso LightGBM (w)')
plt.ylabel('PR-AUC en validación')
plt.title('Búsqueda de peso óptimo para ensemble')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

yprob_val = best_w * yprob_lgb_val + (1 - best_w) * yprob_xgb_val
yprob_test = best_w * yprob_lgb_test + (1 - best_w) * yprob_xgb_test

### Métricas en validación

In [ ]:
results_val = {'PR-AUC': best_prauc, 'AUC-ROC': roc_auc_score(y_val, yprob_val)}
prec, rec, _ = precision_recall_curve(y_val, yprob_val)
pr90 = 0.0
for p, r in zip(prec[:-1], rec[:-1]):
    if r >= 0.90 and p > pr90:
        pr90 = p
results_val['Prec@Rec90'] = pr90

print('Métricas en validación:')
for k, v in results_val.items():
    print(f'  {k}: {v:.4f}')

## 8. Selección de Threshold (F2)

Threshold F2 sobre validación.

In [ ]:
target_precision = 0.60
thrs = np.linspace(0.01, 0.99, 500)
best_t_60 = None; best_rec_60 = -1; best_prec_60 = 0.0
results_table = []

for t in thrs:
    yt = (yprob_val >= t).astype(int)
    p = precision_score(y_val, yt, zero_division=0)
    r = recall_score(y_val, yt, zero_division=0)
    results_table.append({'threshold': t, 'precision': p, 'recall': r})
    if p >= target_precision and r > best_rec_60:
        best_t_60, best_prec_60, best_rec_60 = t, p, r

results_df = pd.DataFrame(results_table)
results_df['f2'] = (5 * results_df['precision'] * results_df['recall']) / (4 * results_df['precision'] + results_df['recall'] + 1e-10)
results_df['f1'] = (2 * results_df['precision'] * results_df['recall']) / (results_df['precision'] + results_df['recall'] + 1e-10)
best_f2_row = results_df.loc[results_df['f2'].idxmax()]
best_f1_row = results_df.loc[results_df['f1'].idxmax()]

print('=' * 70)
if best_rec_60 > 0:
    print(f'Threshold precision >= {target_precision:.0%}: t={best_t_60:.4f}, Prec={best_prec_60:.4f}, Rec={best_rec_60:.4f}')
    if best_rec_60 < 0.10:
        print('>> Recall inutilizable. Usando F2.')
else:
    print(f'NO SE ALCANZA precision >= {target_precision:.0%}.')

print(f'Threshold F2: t={best_f2_row["threshold"]:.4f}, Prec={best_f2_row["precision"]:.4f}, Rec={best_f2_row["recall"]:.4f}, F2={best_f2_row["f2"]:.4f}')
print(f'Threshold F1: t={best_f1_row["threshold"]:.4f}, Prec={best_f1_row["precision"]:.4f}, Rec={best_f1_row["recall"]:.4f}, F1={best_f1_row["f1"]:.4f}')

best_t = best_f2_row['threshold']
best_prec = best_f2_row['precision']
best_rec = best_f2_row['recall']
print(f'\nThreshold FINAL (F2): t={best_t:.4f}, Prec={best_prec:.4f}, Rec={best_rec:.4f}')
print('=' * 70)

### Curva Precision-Recall

In [ ]:
prec_curve, rec_curve, _ = precision_recall_curve(y_val, yprob_val)
plt.figure(figsize=(10, 6))
plt.plot(rec_curve, prec_curve, 'b-', lw=2, label=f'Ensemble (PR-AUC={average_precision_score(y_val, yprob_val):.4f})')
plt.axhline(y=target_precision, color='r', linestyle='--', alpha=0.5, label=f'Target precision={target_precision}')
plt.scatter([best_rec], [best_prec], color='darkgreen', s=100, zorder=5,
            label=f'Thr={best_t:.3f} (P={best_prec:.3f}, R={best_rec:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'Curva PR — Ensemble {LABEL}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
n_rows = len(results_df)
step = max(1, n_rows // 20)
print('Thresholds:')
print(results_df[::step].round(4).to_string(index=False))

## 9. Evaluación Final en Test

In [ ]:
yp_test = (yprob_test >= best_t).astype(int)

print('=' * 60)
print(f'EVALUACIÓN FINAL EN TEST — {LABEL}')
print('=' * 60)
print(f'Modelo: Ensemble (LightGBM + XGBoost, w={best_w:.3f})')
print(f'Threshold: {best_t:.4f}')
print()
print(classification_report(y_test, yp_test, digits=4))
print()

cm = confusion_matrix(y_test, yp_test)
print('Matriz de confusión:')
print(f'              TN={cm[0,0]:,}   FP={cm[0,1]:,}')
print(f'              FN={cm[1,0]:,}   TP={cm[1,1]:,}')
print()
print(f'Fraudes detectados (TP):  {cm[1,1]:,} / {cm[1,0]+cm[1,1]:,} ({cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%)')
print(f'Falsos positivos (FP):    {cm[0,1]:,}')
print(f'Alertas totales:          {cm[0,1]+cm[1,1]:,}')
print(f'Precisión en test:        {cm[1,1]/(cm[0,1]+cm[1,1])*100:.1f}%')
print(f'Recall en test:           {cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%')

In [ ]:
print('\n' + '=' * 60)
print('MÉTRICAS COMPLETAS')
print('=' * 60)
print(f'  PR-AUC (test):          {average_precision_score(y_test, yprob_test):.4f}')
print(f'  AUC-ROC (test):         {roc_auc_score(y_test, yprob_test):.4f}')
print(f'  F1-score (fraude):      {f1_score(y_test, yp_test):.4f}')
print(f'  Precision@Recall=0.9:   {results_val["Prec@Rec90"]:.4f}')
print(f'  Alertas/100k:           {(yp_test.sum() / len(yp_test) * 100000):.1f}')
print()
pauc_ens = average_precision_score(y_test, yprob_test)
print(f'Comparativa:')
print(f'  LightGBM solo:  PR-AUC={average_precision_score(y_test, yprob_lgb_test):.4f}')
print(f'  XGBoost solo:   PR-AUC={average_precision_score(y_test, yprob_xgb_test):.4f}')
print(f'  Ensemble:       PR-AUC={pauc_ens:.4f}')
print()
print(f'Threshold recomendado:    {best_t:.4f}')

## 10. Feature Importance

In [ ]:
imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.booster_.feature_importance(importance_type='gain')
})
imp = imp.sort_values('importance', ascending=False).head(20)
imp['importance_pct'] = imp['importance'] / imp['importance'].sum() * 100

plt.figure(figsize=(10, 8))
sns.barplot(data=imp, y='feature', x='importance_pct', palette='viridis')
plt.xlabel('Importancia relativa (%)')
plt.ylabel('Feature')
plt.title(f'Top 20 Features — LightGBM ({LABEL})')
plt.tight_layout()
plt.show()

## 11. Conclusiones

Ejecutar la siguiente celda para ver el resumen final con valores reales.

In [ ]:
final_prauc = average_precision_score(y_test, yprob_test)
final_auc = roc_auc_score(y_test, yprob_test)
final_prec = cm[1,1]/(cm[0,1]+cm[1,1])*100 if (cm[0,1]+cm[1,1]) > 0 else 0
final_rec = cm[1,1]/(cm[1,0]+cm[1,1])*100 if (cm[1,0]+cm[1,1]) > 0 else 0
final_f1 = f1_score(y_test, yp_test)
alertas_100k = yp_test.sum() / len(yp_test) * 100000

print('=' * 60)
print('CONCLUSIONES — Pipeline Completo')
print('=' * 60)
print(f'  Dataset:             {DATASET}')
print(f'  Modelo:              Ensemble LightGBM+XGBoost (w={best_w:.3f})')
print(f'  Threshold:           {best_t:.4f} (F2)')
print(f'  PR-AUC (test):       {final_prauc:.4f}')
print(f'  AUC-ROC (test):      {final_auc:.4f}')
print(f'  Precisión (test):    {final_prec:.1f}%')
print(f'  Recall (test):       {final_rec:.1f}%')
print(f'  F1 (fraude):         {final_f1:.4f}')
print(f'  Alertas / 100k:      {alertas_100k:.1f}')
print('=' * 60)
print('Para cambiar entre v1 y v2, modificar DATASET en la celda de configuración.')